# 05 — Advanced Models

**Project**: [Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines](https://www.drivendata.org/competitions/66/flu-shot-learning/)

**Author**: Jarret Angbazo

**Date**: May 2026

> **Purpose**: LightGBM as the primary model, XGBoost for comparison, hyperparameter tuning,
> feature set ablation, ensemble, and final competition submission.
>
> **Competition metric**: Mean ROC-AUC averaged across both targets.
> **Floor to beat**: See `baseline_results.csv` from `04_baseline_models.ipynb`.

---

## Table of Contents

1. [Setup & Load](#1-setup--load)
2. [LightGBM — Default Parameters](#2-lightgbm--default-parameters)
3. [Feature Set Ablation](#3-feature-set-ablation)
4. [LightGBM — Hyperparameter Tuning](#4-lightgbm--hyperparameter-tuning)
5. [XGBoost — Comparison Model](#5-xgboost--comparison-model)
6. [Logistic Regression — Tuned C](#6-logistic-regression--tuned-c)
7. [Ensemble — Weighted Average](#7-ensemble--weighted-average)
8. [Final Model Selection](#8-final-model-selection)
9. [Competition Submission](#9-competition-submission)
10. [Summary](#10-summary)

---

## 1. Setup & Load <a id='1-setup--load'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from pathlib import Path
from datetime import datetime

import lightgbm as lgb
import xgboost as xgb

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, loguniform

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
%matplotlib inline
sns.set_style('whitegrid')

# ── Paths ──────────────────────────────────────────────────────────────────
PROCESSED = Path('../data/processed')
MODELS    = Path('../models');    MODELS.mkdir(exist_ok=True)
FIGURES   = Path('../reports/figures'); FIGURES.mkdir(exist_ok=True)
PREDS     = Path('../data/predictions'); PREDS.mkdir(exist_ok=True)

ID_COL      = 'respondent_id'
TARGET_COLS = ['h1n1_vaccine', 'seasonal_vaccine']
RANDOM_SEED = 42

In [ ]:
X_train = pd.read_csv(PROCESSED / 'X_train_features.csv', index_col=ID_COL)
X_test  = pd.read_csv(PROCESSED / 'X_test_features.csv',  index_col=ID_COL)
y_train = pd.read_csv(PROCESSED / 'y_train.csv',          index_col=ID_COL)
baseline_results = pd.read_csv(MODELS / 'baseline_results.csv')

baseline_to_beat = baseline_results['avg_auc'].max()
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nBaseline results:")
print(baseline_results[['model','avg_auc']].sort_values('avg_auc', ascending=False).to_string(index=False))
print(f"\nAvg AUC floor to beat: {baseline_to_beat:.4f}")

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Leaderboard for this notebook
leaderboard = []

def cv_auc_pair(models_dict, X, y_df, cv=CV, verbose=True):
    """CV ROC-AUC for a {target: estimator} dict. Returns avg AUC."""
    aucs = []
    results = {}
    for t, est in models_dict.items():
        scores = cross_val_score(est, X, y_df[t], cv=cv, scoring='roc_auc', n_jobs=-1)
        results[t] = {'mean': scores.mean(), 'std': scores.std()}
        aucs.append(scores.mean())
        if verbose:
            print(f"  {t:<25s}  AUC = {scores.mean():.4f} ± {scores.std():.4f}")
    avg = np.mean(aucs)
    if verbose:
        print(f"  {'AVG':<25s}  AUC = {avg:.4f}")
    return avg, results

## 2. LightGBM — Default Parameters <a id='2-lightgbm--default-parameters'></a>

**Why LightGBM is the primary model (from EDA/FE analysis):**
- Handles mixed ordinal/binary features natively
- `scale_pos_weight` parameter controls class imbalance without resampling
- Leaf-wise tree growth captures interaction effects (doctor_recc × opinion)
- Fast enough for repeated CV during tuning

Two separate binary classifiers — one per target.

In [ ]:
lgb_h1n1_default = lgb.LGBMClassifier(
    n_estimators     = 500,
    learning_rate    = 0.05,
    num_leaves       = 31,          # default
    min_child_samples= 20,          # default
    scale_pos_weight = 3.71,        # h1n1 imbalance ratio (from EDA)
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
    verbose          = -1,
)

lgb_seas_default = lgb.LGBMClassifier(
    n_estimators     = 500,
    learning_rate    = 0.05,
    num_leaves       = 31,
    min_child_samples= 20,
    scale_pos_weight = 1.0,         # seasonal near-balanced
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
    verbose          = -1,
)

print("LightGBM — default parameters:")
avg_lgb_default, lgb_default_results = cv_auc_pair(
    {'h1n1_vaccine': lgb_h1n1_default, 'seasonal_vaccine': lgb_seas_default},
    X_train, y_train
)
leaderboard.append({'model': 'LightGBM (default)', 'avg_auc': avg_lgb_default,
                    **{t: lgb_default_results[t]['mean'] for t in TARGET_COLS}})

improvement = avg_lgb_default - baseline_to_beat
print(f"\n  vs. best baseline: {improvement:+.4f}")

## 3. Feature Set Ablation <a id='3-feature-set-ablation'></a>

Tests three feature sets recommended in `03_feature_engineering.ipynb`:

- **Set A**: Cleaned features only (no engineered features)
- **Set B**: Set A + missingness indicators + `high_vacc_ind_occ` + `doctor_recc_both`
- **Set C** (full): Set B + composites (`opinion_*_composite`, `behavior_composite`, `missing_opinion_count`)

This tells us how much each engineering layer actually adds.

In [ ]:
# Identify engineered columns created in 03_feature_engineering
ENGINEERED_COLS = [
    'missing_opinion_count',
    'high_vacc_ind_occ',
    'doctor_recc_both',
    'opinion_h1n1_composite',
    'opinion_seas_composite',
    'behavior_composite',
]
INDICATOR_COLS = [c for c in X_train.columns if c.endswith('_missing')]

# Set A: no engineered features, no indicators
set_a_cols = [c for c in X_train.columns
              if c not in ENGINEERED_COLS and c not in INDICATOR_COLS]

# Set B: + indicators + high_vacc_ind_occ + doctor_recc_both
set_b_extra = INDICATOR_COLS + ['high_vacc_ind_occ', 'doctor_recc_both']
set_b_cols  = set_a_cols + [c for c in set_b_extra if c in X_train.columns]

# Set C: full
set_c_cols  = list(X_train.columns)

print(f"Set A: {len(set_a_cols)} features")
print(f"Set B: {len(set_b_cols)} features  (+{len(set_b_cols)-len(set_a_cols)} engineered)")
print(f"Set C: {len(set_c_cols)} features  (+{len(set_c_cols)-len(set_b_cols)} composites)")

def lgb_default_pair():
    return {
        'h1n1_vaccine'   : lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                                scale_pos_weight=3.71, random_state=RANDOM_SEED,
                                                n_jobs=-1, verbose=-1),
        'seasonal_vaccine': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                                scale_pos_weight=1.0, random_state=RANDOM_SEED,
                                                n_jobs=-1, verbose=-1),
    }

ablation_results = {}
for label, cols in [('Set A (base)', set_a_cols), ('Set B (+indicators)', set_b_cols), ('Set C (full)', set_c_cols)]:
    print(f"\n{label}:")
    avg, res = cv_auc_pair(lgb_default_pair(), X_train[cols], y_train)
    ablation_results[label] = {'avg_auc': avg, 'n_features': len(cols), **{t: res[t]['mean'] for t in TARGET_COLS}}

print("\n=== ABLATION SUMMARY ===")
abl_df = pd.DataFrame(ablation_results).T[['n_features','h1n1_vaccine','seasonal_vaccine','avg_auc']]
print(abl_df.round(4).to_string())

# Use the best-performing feature set going forward
best_set_label = max(ablation_results, key=lambda k: ablation_results[k]['avg_auc'])
if 'A' in best_set_label:
    FEATURE_COLS = set_a_cols
elif 'B' in best_set_label:
    FEATURE_COLS = set_b_cols
else:
    FEATURE_COLS = set_c_cols

print(f"\nUsing: {best_set_label} ({len(FEATURE_COLS)} features) for tuning")

## 4. LightGBM — Hyperparameter Tuning <a id='4-lightgbm--hyperparameter-tuning'></a>

RandomizedSearchCV over the key LightGBM hyperparameters. Tuned independently per target
because their optimal configurations may differ (h1n1 is harder, more imbalanced).

Key parameters to tune:
- `num_leaves` — controls model complexity (main LightGBM param, like max_depth)
- `min_child_samples` — regularisation via minimum leaf size
- `feature_fraction` — column subsampling (reduces overfitting, speeds up training)
- `bagging_fraction` / `bagging_freq` — row subsampling
- `learning_rate` + `n_estimators` — stepped together via early stopping logic

In [ ]:
param_dist = {
    'num_leaves'       : randint(20, 150),
    'min_child_samples': randint(10, 100),
    'feature_fraction' : [0.6, 0.7, 0.8, 0.9, 1.0],
    'bagging_fraction' : [0.7, 0.8, 0.9, 1.0],
    'bagging_freq'     : [0, 1, 5],
    'learning_rate'    : loguniform(0.01, 0.2),
    'n_estimators'     : [300, 500, 750, 1000],
    'reg_alpha'        : loguniform(1e-4, 1.0),
    'reg_lambda'       : loguniform(1e-4, 1.0),
}

# ── h1n1 tuning ───────────────────────────────────────────────────────────
print("Tuning LightGBM — h1n1_vaccine ...")
lgb_h1n1_base = lgb.LGBMClassifier(
    scale_pos_weight=3.71,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=-1,
)
search_h1n1 = RandomizedSearchCV(
    lgb_h1n1_base,
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=CV,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=0,
)
search_h1n1.fit(X_train[FEATURE_COLS], y_train['h1n1_vaccine'])
print(f"  Best h1n1 AUC: {search_h1n1.best_score_:.4f}")
print(f"  Best params  : {search_h1n1.best_params_}")

In [ ]:
# ── seasonal tuning ───────────────────────────────────────────────────────
print("Tuning LightGBM — seasonal_vaccine ...")
lgb_seas_base = lgb.LGBMClassifier(
    scale_pos_weight=1.0,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=-1,
)
search_seas = RandomizedSearchCV(
    lgb_seas_base,
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=CV,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=0,
)
search_seas.fit(X_train[FEATURE_COLS], y_train['seasonal_vaccine'])
print(f"  Best seasonal AUC: {search_seas.best_score_:.4f}")
print(f"  Best params      : {search_seas.best_params_}")

In [ ]:
# Tuned models — best estimators from search
lgb_h1n1_tuned = search_h1n1.best_estimator_
lgb_seas_tuned = search_seas.best_estimator_

avg_tuned, tuned_results = cv_auc_pair(
    {'h1n1_vaccine': lgb_h1n1_tuned, 'seasonal_vaccine': lgb_seas_tuned},
    X_train[FEATURE_COLS], y_train
)
leaderboard.append({'model': 'LightGBM (tuned)', 'avg_auc': avg_tuned,
                    **{t: tuned_results[t]['mean'] for t in TARGET_COLS}})

print(f"\n  Default LightGBM avg AUC : {avg_lgb_default:.4f}")
print(f"  Tuned   LightGBM avg AUC : {avg_tuned:.4f}")
print(f"  Tuning improvement       : {avg_tuned - avg_lgb_default:+.4f}")

# Save
lgb_h1n1_tuned.fit(X_train[FEATURE_COLS], y_train['h1n1_vaccine'])
lgb_seas_tuned.fit(X_train[FEATURE_COLS], y_train['seasonal_vaccine'])
joblib.dump({'h1n1': lgb_h1n1_tuned, 'seasonal': lgb_seas_tuned,
             'feature_cols': FEATURE_COLS}, MODELS / 'lgb_tuned.pkl')
print("\nModels saved: lgb_tuned.pkl")

## 5. XGBoost — Comparison Model <a id='5-xgboost--comparison-model'></a>

XGBoost as a second gradient boosting implementation. Uses the same feature set as the
best-performing LightGBM configuration. `eval_metric='auc'` aligns with competition metric.

In [ ]:
xgb_h1n1 = xgb.XGBClassifier(
    n_estimators   = 500,
    learning_rate  = 0.05,
    max_depth      = 5,
    subsample      = 0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3.71,
    eval_metric    = 'auc',
    use_label_encoder=False,
    random_state   = RANDOM_SEED,
    n_jobs         = -1,
    verbosity      = 0,
)

xgb_seas = xgb.XGBClassifier(
    n_estimators   = 500,
    learning_rate  = 0.05,
    max_depth      = 5,
    subsample      = 0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.0,
    eval_metric    = 'auc',
    use_label_encoder=False,
    random_state   = RANDOM_SEED,
    n_jobs         = -1,
    verbosity      = 0,
)

print("XGBoost (default parameters):")
avg_xgb, xgb_results = cv_auc_pair(
    {'h1n1_vaccine': xgb_h1n1, 'seasonal_vaccine': xgb_seas},
    X_train[FEATURE_COLS], y_train
)
leaderboard.append({'model': 'XGBoost (default)', 'avg_auc': avg_xgb,
                    **{t: xgb_results[t]['mean'] for t in TARGET_COLS}})

xgb_h1n1.fit(X_train[FEATURE_COLS], y_train['h1n1_vaccine'])
xgb_seas.fit(X_train[FEATURE_COLS], y_train['seasonal_vaccine'])
joblib.dump({'h1n1': xgb_h1n1, 'seasonal': xgb_seas,
             'feature_cols': FEATURE_COLS}, MODELS / 'xgb_default.pkl')

## 6. Logistic Regression — Tuned C <a id='6-logistic-regression--tuned-c'></a>

LR with regularisation strength `C` tuned over a log-uniform grid. Included because the linear
signal in this dataset is strong — tuned LR sometimes competes with tree models on survey data.

In [ ]:
from sklearn.model_selection import GridSearchCV

c_grid = {'clf__C': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]}

for t, weight in [('h1n1_vaccine', 'balanced'), ('seasonal_vaccine', None)]:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=2000, class_weight=weight,
                                      solver='lbfgs', random_state=RANDOM_SEED)),
    ])
    gs = GridSearchCV(pipe, c_grid, scoring='roc_auc', cv=CV, n_jobs=-1)
    gs.fit(X_train[FEATURE_COLS], y_train[t])
    print(f"{t}: best C={gs.best_params_['clf__C']}  AUC={gs.best_score_:.4f}")

# Run both together for leaderboard
best_c_h1n1 = GridSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                         solver='lbfgs', random_state=RANDOM_SEED))]),
    c_grid, scoring='roc_auc', cv=CV, n_jobs=-1
)
best_c_seas = GridSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('clf', LogisticRegression(max_iter=2000, class_weight=None,
                                         solver='lbfgs', random_state=RANDOM_SEED))]),
    c_grid, scoring='roc_auc', cv=CV, n_jobs=-1
)
best_c_h1n1.fit(X_train[FEATURE_COLS], y_train['h1n1_vaccine'])
best_c_seas.fit(X_train[FEATURE_COLS], y_train['seasonal_vaccine'])

avg_lr_tuned = np.mean([best_c_h1n1.best_score_, best_c_seas.best_score_])
leaderboard.append({'model': 'Logistic Regression (tuned C)',
                    'h1n1_vaccine': best_c_h1n1.best_score_,
                    'seasonal_vaccine': best_c_seas.best_score_,
                    'avg_auc': avg_lr_tuned})
print(f"\nLR tuned avg AUC: {avg_lr_tuned:.4f}")

## 7. Ensemble — Weighted Average <a id='7-ensemble--weighted-average'></a>

Simple weighted average of LightGBM (tuned), XGBoost, and Random Forest OOF probabilities.
More flexible than a VotingClassifier because weights can be optimised per target.

**OOF (out-of-fold) approach**: train each model on 4 folds, predict on the held-out fold.
Average OOF predictions approximate test performance without a separate hold-out set.

In [ ]:
from sklearn.base import clone

def oof_probas(estimator, X, y, cv=CV):
    """Return out-of-fold predicted probabilities."""
    oof = np.zeros(len(y))
    for train_idx, val_idx in cv.split(X, y):
        model = clone(estimator)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    return oof

# Reload baseline RF
rf_models = joblib.load(MODELS / 'baseline_random_forest.pkl')

oof_probas_h1n1 = {}
oof_probas_seas  = {}

print("Computing OOF probabilities...")
for name, est_h1n1, est_seas in [
    ('lgb_tuned', lgb_h1n1_tuned, lgb_seas_tuned),
    ('xgb',       xgb_h1n1,       xgb_seas),
    ('rf',        rf_models['h1n1'], rf_models['seasonal']),
]:
    print(f"  {name}...")
    oof_probas_h1n1[name] = oof_probas(est_h1n1, X_train[FEATURE_COLS], y_train['h1n1_vaccine'])
    oof_probas_seas[name]  = oof_probas(est_seas,  X_train[FEATURE_COLS], y_train['seasonal_vaccine'])
    print(f"    h1n1    AUC: {roc_auc_score(y_train['h1n1_vaccine'],    oof_probas_h1n1[name]):.4f}")
    print(f"    seasonal AUC: {roc_auc_score(y_train['seasonal_vaccine'], oof_probas_seas[name]):.4f}")

In [ ]:
from scipy.optimize import minimize

def neg_avg_auc(weights, oof_dict, y_h1n1, y_seas):
    """Objective: negative avg AUC for two-target ensemble."""
    w = np.array(weights)
    w = np.clip(w, 0, None)
    w = w / w.sum()

    names = list(oof_dict[0].keys())
    blend_h1n1 = sum(w[i] * oof_dict[0][n] for i, n in enumerate(names))
    blend_seas  = sum(w[i] * oof_dict[1][n] for i, n in enumerate(names))

    auc_h1n1 = roc_auc_score(y_h1n1, blend_h1n1)
    auc_seas  = roc_auc_score(y_seas,  blend_seas)
    return -(auc_h1n1 + auc_seas) / 2

n_models = len(oof_probas_h1n1)
init_weights = np.ones(n_models) / n_models

result = minimize(
    neg_avg_auc,
    x0=init_weights,
    args=([oof_probas_h1n1, oof_probas_seas],
          y_train['h1n1_vaccine'], y_train['seasonal_vaccine']),
    method='Nelder-Mead',
    options={'maxiter': 1000},
)

opt_weights = np.clip(result.x, 0, None)
opt_weights /= opt_weights.sum()
model_names = list(oof_probas_h1n1.keys())

print("Optimal ensemble weights:")
for name, w in zip(model_names, opt_weights):
    print(f"  {name}: {w:.3f}")

# Evaluate ensemble OOF
blend_h1n1_oof = sum(opt_weights[i] * oof_probas_h1n1[n] for i, n in enumerate(model_names))
blend_seas_oof  = sum(opt_weights[i] * oof_probas_seas[n]  for i, n in enumerate(model_names))

ens_h1n1_auc = roc_auc_score(y_train['h1n1_vaccine'],    blend_h1n1_oof)
ens_seas_auc  = roc_auc_score(y_train['seasonal_vaccine'], blend_seas_oof)
ens_avg_auc   = (ens_h1n1_auc + ens_seas_auc) / 2

print(f"\nEnsemble OOF AUC:")
print(f"  h1n1_vaccine    : {ens_h1n1_auc:.4f}")
print(f"  seasonal_vaccine: {ens_seas_auc:.4f}")
print(f"  AVG             : {ens_avg_auc:.4f}")

leaderboard.append({'model': 'Ensemble (weighted avg)',
                    'h1n1_vaccine': ens_h1n1_auc, 'seasonal_vaccine': ens_seas_auc,
                    'avg_auc': ens_avg_auc})
joblib.dump({'weights': opt_weights, 'model_names': model_names,
             'feature_cols': FEATURE_COLS}, MODELS / 'ensemble_weights.pkl')

## 8. Final Model Selection <a id='8-final-model-selection'></a>

In [ ]:
# Include baseline results
lb_baseline = pd.read_csv(MODELS / 'baseline_results.csv')
lb_advanced = pd.DataFrame(leaderboard)
lb_all      = pd.concat([lb_baseline, lb_advanced], ignore_index=True)
lb_all      = lb_all.sort_values('avg_auc', ascending=False).reset_index(drop=True)
lb_all[['h1n1_vaccine','seasonal_vaccine','avg_auc']] = lb_all[['h1n1_vaccine','seasonal_vaccine','avg_auc']].round(4)

print("=== FULL LEADERBOARD ===")
print(lb_all[['model','h1n1_vaccine','seasonal_vaccine','avg_auc']].to_string(index=False))
lb_all.to_csv(MODELS / 'full_leaderboard.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(lb_all))]
ax.barh(lb_all['model'], lb_all['avg_auc'], color=colors, edgecolor='white')
ax.axvline(0.5, color='grey', linestyle='--', linewidth=1, alpha=0.7, label='Random (0.5)')
ax.set_xlabel('Mean ROC-AUC (5-fold CV)')
ax.set_title('Full Model Leaderboard — Avg AUC Across Both Targets', fontsize=12, pad=10)
ax.legend(fontsize=9)
ax.set_xlim(0.45, 1.0)
for i, (_, row) in enumerate(lb_all.iterrows()):
    ax.text(row['avg_auc'] + 0.002, i, f"{row['avg_auc']:.4f}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'advanced_full_leaderboard.png', dpi=150)
plt.show()

best_model_name = lb_all.iloc[0]['model']
print(f"\nSelected model: {best_model_name}")

## 9. Competition Submission <a id='9-competition-submission'></a>

**Submission format** (DrivenData): CSV with columns `respondent_id`, `h1n1_vaccine`,
`seasonal_vaccine`. Values are **probabilities** (not 0/1 labels) — AUC is computed on
continuous scores, not binary predictions.

In [ ]:
# Determine which model to use for submission based on leaderboard
best_name = lb_all.iloc[0]['model']
print(f"Generating submission from: {best_name}")

if 'Ensemble' in best_name:
    # Ensemble: average test predictions from each constituent model
    ens_info = joblib.load(MODELS / 'ensemble_weights.pkl')
    w = ens_info['weights']
    names = ens_info['model_names']
    fc    = ens_info['feature_cols']

    test_h1n1_preds = {}
    test_seas_preds  = {}

    for name, est_h1n1, est_seas in [
        ('lgb_tuned', lgb_h1n1_tuned, lgb_seas_tuned),
        ('xgb',       xgb_h1n1,       xgb_seas),
        ('rf',        rf_models['h1n1'], rf_models['seasonal']),
    ]:
        test_h1n1_preds[name] = est_h1n1.predict_proba(X_test[fc])[:, 1]
        test_seas_preds[name]  = est_seas.predict_proba(X_test[fc])[:, 1]

    final_h1n1 = sum(w[i] * test_h1n1_preds[n] for i, n in enumerate(names))
    final_seas  = sum(w[i] * test_seas_preds[n]  for i, n in enumerate(names))

elif 'LightGBM (tuned)' in best_name:
    fc = FEATURE_COLS
    final_h1n1 = lgb_h1n1_tuned.predict_proba(X_test[fc])[:, 1]
    final_seas  = lgb_seas_tuned.predict_proba(X_test[fc])[:, 1]

elif 'XGBoost' in best_name:
    fc = FEATURE_COLS
    final_h1n1 = xgb_h1n1.predict_proba(X_test[fc])[:, 1]
    final_seas  = xgb_seas.predict_proba(X_test[fc])[:, 1]

else:
    # Fallback: tuned LightGBM
    fc = FEATURE_COLS
    final_h1n1 = lgb_h1n1_tuned.predict_proba(X_test[fc])[:, 1]
    final_seas  = lgb_seas_tuned.predict_proba(X_test[fc])[:, 1]

submission = pd.DataFrame({
    'respondent_id'  : X_test.index,
    'h1n1_vaccine'   : final_h1n1,
    'seasonal_vaccine': final_seas,
})

ts = datetime.now().strftime('%Y%m%d_%H%M')
fname = PREDS / f'submission_{ts}.csv'
submission.to_csv(fname, index=False)

print(f"Saved: {fname}")
print(f"Shape: {submission.shape}")
print(f"\nProbability ranges:")
print(f"  h1n1_vaccine   : [{submission['h1n1_vaccine'].min():.3f}, {submission['h1n1_vaccine'].max():.3f}]")
print(f"  seasonal_vaccine: [{submission['seasonal_vaccine'].min():.3f}, {submission['seasonal_vaccine'].max():.3f}]")
print(f"\nFirst 5 rows:")
print(submission.head())

## 10. Summary <a id='10-summary'></a>

In [ ]:
best = lb_all.iloc[0]
print("=" * 60)
print(f"  BEST MODEL              : {best['model']}")
print(f"  h1n1_vaccine AUC        : {best['h1n1_vaccine']:.4f}")
print(f"  seasonal_vaccine AUC    : {best['seasonal_vaccine']:.4f}")
print(f"  AVG AUC (competition)   : {best['avg_auc']:.4f}")
print(f"  Improvement over baseline: {best['avg_auc'] - baseline_to_beat:+.4f}")
print("=" * 60)

### What Goes into `06_model_evaluation.ipynb`

- `lgb_tuned.pkl` — primary model for SHAP / permutation importance
- `full_leaderboard.csv` — complete model comparison
- `submission_*.csv` — final competition file
- `FEATURE_COLS` list (embedded in pkl) — ensures evaluation uses same features